In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

print("Распаковываем фотографии")
os.makedirs('/content/photos', exist_ok=True)
!unzip -q "/content/drive/MyDrive/Москва.zip" -d "/content/photos/"
print("Фотографии успешно распакованы!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Распаковываем фотографии...
✅ Фотографии успешно распакованы!


In [ ]:
!pip install -q pandas numpy fastparquet sentence-transformers pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# --- 1. НАСТРОЙКИ ---
DATA_PATH = "/content/temp.parquet"
PHOTOS_DIR = "/content/photos/Москва"
OUTPUT_PATH = "/content/features_with_clip.parquet"

MODEL_NAME = "clip-ViT-B-32"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

model = SentenceTransformer(MODEL_NAME, device=device)

# --- 2. ФУНКЦИЯ ОБРАБОТКИ ОДНОЙ КВАРТИРЫ ---
def get_average_embedding(flat_id: str) -> np.ndarray:
    """Находит все фото квартиры, пропускает через CLIP и усредняет вектор."""
    clean_id = str(flat_id).strip()
    if clean_id.endswith('.0'):
        clean_id = clean_id[:-2]

    flat_dir = os.path.join(PHOTOS_DIR, clean_id)
    if not os.path.exists(flat_dir):
        return None

    images = []
    for filename in os.listdir(flat_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(flat_dir, filename)
            try:
                img = Image.open(img_path).convert("RGB")
                images.append(img)
            except Exception as e:
                print(f"Ошибка чтения {img_path}: {e}")

    if not images:
        return None

    with torch.no_grad():
        img_embeddings = model.encode(images, convert_to_numpy=True, show_progress_bar=False)

    # УСРЕДНЯЕМ ВЕКТОРЫ
    avg_embedding = np.mean(img_embeddings, axis=0)

    return avg_embedding

# --- 3. ОСНОВНОЙ ПРОЦЕСС ---
def main():
    print(f"📄 Загружаем датасет из {DATA_PATH}...")
    df = pd.read_parquet(DATA_PATH)
    df_subset = df.copy()

    print(f"Начинаем извлечение эмбеддингов для {len(df_subset)} квартир")

    all_embeddings = []
    valid_indices = []

    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):
        flat_id = row['flat_id']
        emb = get_average_embedding(flat_id)

        if emb is not None:
            all_embeddings.append(emb)
            valid_indices.append(idx)

    if not all_embeddings:
        print("ОШИБКА: Не удалось найти/обработать ни одной фотографии")
        return

    # --- 4. ОБОГАЩЕНИЕ ДАТАСЕТА ---
    print(f"\nПриклеиваем {len(all_embeddings[0])} новых колонок к датасету...")

    df_final = df_subset.loc[valid_indices].copy()
    emb_matrix = np.vstack(all_embeddings)
    emb_cols = [f"clip_emb_{i}" for i in range(emb_matrix.shape[1])]
    emb_df = pd.DataFrame(emb_matrix, columns=emb_cols, index=df_final.index)

    df_final = pd.concat([df_final, emb_df], axis=1)
    df_final = df_final.drop(columns=['flat_id'])
    df_final.to_parquet(OUTPUT_PATH, index=False, engine='fastparquet')
    print(f"Обогащенный датасет ({df_final.shape[0]} строк, {df_final.shape[1]} колонок) сохранен в {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

🚀 Используем устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

📄 Загружаем датасет из /content/temp.parquet...
📸 Начинаем извлечение эмбеддингов для 9280 квартир...


  0%|          | 0/9280 [00:00<?, ?it/s]

Ошибка чтения /content/photos/Москва/358901536/5.jpg: cannot identify image file '/content/photos/Москва/358901536/5.jpg'
Ошибка чтения /content/photos/Москва/358901536/6.jpg: cannot identify image file '/content/photos/Москва/358901536/6.jpg'
Ошибка чтения /content/photos/Москва/351057897/3.jpg: cannot identify image file '/content/photos/Москва/351057897/3.jpg'
Ошибка чтения /content/photos/Москва/351953273/5.jpg: cannot identify image file '/content/photos/Москва/351953273/5.jpg'
Ошибка чтения /content/photos/Москва/356437220/2.jpg: cannot identify image file '/content/photos/Москва/356437220/2.jpg'
Ошибка чтения /content/photos/Москва/351450269/5.jpg: cannot identify image file '/content/photos/Москва/351450269/5.jpg'
Ошибка чтения /content/photos/Москва/351153037/2.jpg: cannot identify image file '/content/photos/Москва/351153037/2.jpg'
Ошибка чтения /content/photos/Москва/357900622/6.jpg: cannot identify image file '/content/photos/Москва/357900622/6.jpg'
Ошибка чтения /content/p

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, median_absolute_error, r2_score
from catboost import CatBoostRegressor, Pool

DATA_PATH = "/content/features_with_clip.parquet"
RANDOM_SEED = 42
TEST_SIZE = 0.2

CB_PARAMS = {
    "iterations": 2500,
    "learning_rate": 0.03,
    "depth": 7,
    "eval_metric": "MAPE",
    "random_seed": RANDOM_SEED,
    "od_type": "Iter",
    "od_wait": 100,
    "task_type": "GPU",
}


def load_data(path):
    df = pd.read_parquet(path)
    df["price_mln"] = df["price_total"] / 1_000_000
    if "description_full" in df.columns:
        df["description_full"] = df["description_full"].fillna("Нет").astype(str)
    return df


def prepare_xy(df):
    price_prm2 = df["price_total"] / df["area_total_m2"]
    y = np.log1p(price_prm2)
    drop_cols = ['price_total', 'price_mln', 'price_per_m2_calc', 'target_log_prm2', 'price_log']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    cat = [c for c in ['building_material', 'condition', 'geo_cluster'] if c in X.columns]
    txt = ['description_full'] if 'description_full' in X.columns else []
    return X, y, cat, txt


def get_metrics(y_true_mln, y_pred_log_prm2, area_m2):
    y_pred_total_mln = (np.expm1(y_pred_log_prm2) * area_m2) / 1_000_000
    return {
        "mae": mean_absolute_error(y_true_mln, y_pred_total_mln),
        "medae": median_absolute_error(y_true_mln, y_pred_total_mln),
        "mape": mean_absolute_percentage_error(y_true_mln, y_pred_total_mln) * 100,
        "r2": r2_score(y_true_mln, y_pred_total_mln)
    }


def main():
    df = load_data(DATA_PATH)

    # ВАЛИДАЦИЯ (ЧТОБЫ УЗНАТЬ МЕТРИКИ) ===
    print("ЭТАП 1: Оценка качества...")
    train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED)

    X_train, y_train, cat, txt = prepare_xy(train_df)
    X_test, y_test, _, _ = prepare_xy(test_df)

    val_model = CatBoostRegressor(**CB_PARAMS, verbose=False)
    val_model.fit(Pool(X_train, y_train, cat, txt), eval_set=Pool(X_test, y_test, cat, txt))

    # Считаем и выводим метрики (это реальная сила твоей модели)
    preds = val_model.predict(X_test)
    m = get_metrics(test_df["price_mln"], preds, test_df["area_total_m2"])

    print("\nМЕТРИКИ КАЧЕСТВА (нa отложенной выборке):")
    print(f"   MAPE:  {m['mape']:.2f}%")
    print(f"   MedAE: {m['medae']:.3f} млн руб.")
    print(f"   R2:    {m['r2']:.4f}")
    print(f"   MAE:   {m['mae']:.3f} млн руб.")

    best_iter = val_model.get_best_iteration()

    #(ОБУЧЕНИЕ НА 100% ДАННЫХ)
    print(f"\nЭТАП 2: Обучение финальной модели на всех {len(df)} строках...")
    X_full, y_full, cat, txt = prepare_xy(df)

    prod_params = CB_PARAMS.copy()
    prod_params["iterations"] = best_iter  # Используем лучшее число деревьев
    prod_params.pop("od_wait")
    prod_params.pop("od_type")  # Убираем лишнее

    final_model = CatBoostRegressor(**prod_params, verbose=500)
    final_model.fit(Pool(X_full, y_full, cat, txt))

    # Сохраняем
    os.makedirs("models", exist_ok=True)
    final_model.save_model("models/house_model_with_img.cbm")
    print(f"Метрики зафиксированы, модель обучена на максимуме данных и сохранена.")


if __name__ == "__main__":
    main()

'''МЕТРИКИ КАЧЕСТВА (на отложенной выборке):
   MAPE:  14.85%
   MedAE: 2.364 млн ру'
   R2:    0.8596
   MAE:   8.348 млн руб'''

ЭТАП 1: Оценка качества...

МЕТРИКИ КАЧЕСТВА (на отложенной выборке):
   MAPE:  15.12%
   MedAE: 2.383 млн руб.
   R2:    0.8688
   MAE:   8.748 млн руб.

ЭТАП 2: Обучение финальной модели на всех 7781 строках...
0:	learn: 0.0273768	total: 77.3ms	remaining: 3m 13s
500:	learn: 0.0105936	total: 40s	remaining: 2m 39s
1000:	learn: 0.0092534	total: 1m 19s	remaining: 1m 58s
1500:	learn: 0.0084231	total: 1m 57s	remaining: 1m 17s
2000:	learn: 0.0077905	total: 2m 35s	remaining: 38.6s
2497:	learn: 0.0072799	total: 3m 14s	remaining: 0us
Метрики зафиксированы, модель обучена на максимуме данных и сохранена.


"МЕТРИКИ КАЧЕСТВА (на отложенной выборке):\n   MAPE:  14.85%\n   MedAE: 2.364 млн ру'\n   R2:    0.8596\n   MAE:   8.348 млн руб"